#### Add the gemini API key ( From Google AI Studio )

In [1]:
import os
from google.colab import userdata
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
print("✅ Gemini API key setup complete.")

✅ Gemini API key setup complete.


#### Import ADK Components

In [2]:
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


### 1. Why Multi-Agent Systems?

**The Problem: The "Do-It-All" Agent**

Single agents can do a lot. But what happens when the task gets complex? A single "monolithic" agent that tries to do research, writing, editing, and fact-checking all at once becomes a problem. Its instruction prompt gets long and confusing. It's hard to debug (which part failed?), difficult to maintain, and often produces unreliable results.

**The Solution: A Team of Specialists**

Instead of one "do-it-all" agent, we can build a **multi-agent system**. This is a team of simple, specialized agents that collaborate, just like a real-world team. Each agent has one clear job (e.g., one agent *only* does research, another *only* writes). This makes them easier to build, easier to test, and much more powerful and reliable when working together.

To learn more, check out the documentation related to [LLM agents in ADK](https://google.github.io/adk-docs/agents/llm-agents/).

**Architecture: Single Agent vs Multi-Agent Team**
<img width="800" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/multi-agent-team.png" alt="Multi-agent Team" />


#### Research & Summarization System :
Let's build a system with two specialized agents:

1. **Research Agent** - Searches for information using Google Search
2. **Summarizer Agent** - Creates concise summaries from research findings

In [3]:
# Research Agent: Its job is to use the google_search tool and present findings.

research_agent = Agent(

    name="ResearchAgent",

    description="This agent uses Google Search to find relevant information and presents the findings with citations.",

    model=Gemini(model="gemini-2.5-flash-lite"),

    instruction="""You are a specialized research agent. Your only job is to use the
    google_search tool to find 2-3 pieces of relevant information on the given topic and present the findings with citations.""" ,

    tools=[google_search],

    output_key="research_findings", # The result of this agent will be stored in the session with this key
)

print("Research Agent created.")

Research Agent created.


In [4]:
# Summarizer Agent: Its job is to summarize the text it receives.
summarizer_agent = Agent(

    name="SummarizerAgent",

    description="This agent summarizes research findings into a concise bulleted list.",

    model=Gemini(model="gemini-2.5-flash-lite"),

    # The instruction is modified to request a bulleted list for a clear output format.
    instruction="""Read the provided research findings: {research_findings}
Create a concise summary as a bulleted list with 3-5 key points.""",

    output_key="final_summary",
)

print("Summarizer Agent created.")

Summarizer Agent created.


#### Then we bring the agents together under a root agent (Coordinator)

Here we're using `AgentTool` to wrap the sub-agents to make them callable tools for the root agent.

In [8]:
# Root Coordinator: Orchestrates the workflow by calling the sub-agents as tools.
root_agent = Agent(

    name="ResearchCoordinator",

    model=Gemini(model="gemini-2.5-flash-lite"),

    # This instruction tells the root agent HOW to use its tools (which are the other agents).

    instruction="""You are a research coordinator. Your goal is to answer the user's query by orchestrating a workflow.
1. First, you MUST call the `ResearchAgent` tool to find relevant information on the topic provided by the user.
2. Next, after receiving the research findings, you MUST call the `SummarizerAgent` tool to create a concise summary.
3. Finally, present the final summary clearly to the user as your response in a structured and bulleteed format""",

    # We wrap the sub-agents in `AgentTool` to make them callable tools for the root agent.

    tools=[AgentTool(research_agent), AgentTool(summarizer_agent)],
)

print("✅ root_agent created.")

✅ root_agent created.


In [6]:
runner = InMemoryRunner(agent=root_agent)

In [9]:
response = await runner.run_debug(
    "What are the latest advancements in quantum computing and what do they mean for AI?"
)


 ### Continue session: debug_session_id

User > What are the latest advancements in quantum computing and what do they mean for AI?
ResearchCoordinator > Recent advancements in quantum computing, particularly in creating more stable qubits and enhancing error correction, are paving the way for a powerful integration with artificial intelligence, often referred to as Quantum AI. This synergy promises to overcome the limitations of classical computing for complex AI tasks.

Quantum computers, leveraging qubits that utilize superposition for simultaneous states, offer exponential processing power. This could drastically accelerate AI model training, improve optimization algorithms, and enable more efficient processing of massive datasets. The potential implications for AI include accelerated machine learning, enhanced problem-solving in optimization, more effective data processing, advanced natural language processing, and revolutionary applications in medicine, materials science, and cl

In [10]:
response = await runner.run_debug(
    "What are the latest advancements in quantum computing and what do they mean for AI?"
)


 ### Continue session: debug_session_id

User > What are the latest advancements in quantum computing and what do they mean for AI?
ResearchCoordinator > Quantum computing advancements, particularly in error correction and hardware development, are set to revolutionize AI through a convergence known as Quantum AI (QAI). This integration promises to unlock unprecedented computational power, enabling AI to tackle currently intractable problems.

Key implications for AI include:

*   **Enhanced Processing and Optimization:** Quantum computers can process vast datasets in parallel, speeding up AI tasks like deep learning, leading to faster training and real-time decision-making. They also offer revolutionary optimization capabilities crucial for many AI algorithms.
*   **Improved Machine Learning Models:** Quantum machine learning algorithms can analyze data more effectively, creating more robust AI models. Simultaneously, AI is accelerating quantum computing development, forming a symbio

We've just built your first multi-agent system! we used a single "coordinator" agent to manage the workflow.

‼️ However, **relying on an LLM's instructions to control the order can sometimes be unpredictable.** Next, we'll explore a different pattern that gives you guaranteed, step-by-step execution.

---
---


### 2. Sequential Workflows - The Assembly Line





**The Problem: Unpredictable Order**

The previous multi-agent system worked, but it relied on a **detailed instruction prompt** to force the LLM to run steps in order. This can be unreliable. A complex LLM might decide to skip a step, run them in the wrong order, or get "stuck," making the process unpredictable.

**The Solution: A Fixed Pipeline**

When you need tasks to happen in a **guaranteed, specific order**, you can use a `SequentialAgent`. This agent acts like an assembly line, running each sub-agent in the exact order you list them. The output of one agent automatically becomes the input for the next, creating a predictable and reliable workflow.

**Use Sequential when:** Order matters, you need a linear pipeline, or each step builds on the previous one.

To learn more, check out the documentation related to [sequential agents in ADK](https://google.github.io/adk-docs/agents/workflow-agents/sequential-agents/).



**Architecture: Blog Post Creation Pipeline**
<img width="1000" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/sequential-agent.png" alt="Sequential Agent" />

#### Blog Post Creation with Sequential Agents

Let's build a system with three specialized agents:

1. **Outline Agent** - Creates a blog outline for a given topic
2. **Writer Agent** - Writes a blog post
3. **Editor Agent** - Edits a blog post draft for clarity and structure

In [13]:
# Outline Agent : Creates the initial blog outline

outline_agent = Agent(
    name = "OutlineAgent",
    description = "This agent creates an outline for a blog post.",
    model = Gemini(model="gemini-2.5-flash-lite"),


    instruction ="""Create a blog outline for the given topic with:
    1. A catchy headline
    2. An introduction hook
    3. 3-5 main sections with 2-3 bullet points for each
    4. A concluding thought""",

    output_key = "blog_outline" # The result of this agent will be stored in the session state with this key.
)

print("Outline Agent created.")

Outline Agent created.


In [14]:
# Writer Agent : Writes the whole blog based on the outline from the previous agent

writer_agent = Agent(
    name = "WriterAgent",
    description = "This agent writes a blog post based on an outline.",
    model = Gemini(model="gemini-2.5-flash-lite"),

    # The `{blog_outline}` placeholder automatically injects the state value from the previous agent's output.

    instruction ="""Following this outline strictly: {blog_outline}
    Write a brief, 200 to 300-word blog post with an engaging and informative tone. """,
    output_key = "blog_draft"

)

print("Writer Agent created.")


Writer Agent created.


In [15]:
# Editor Agent: Edits and polishes the draft from the writer agent.
editor_agent = Agent(
    name="EditorAgent",
    model=Gemini(model="gemini-2.5-flash-lite"),
    description="This agent edits and polishes a blog draft.",

    # This agent receives the `{blog_draft}` from the writer agent's output.
    instruction="""Edit this draft: {blog_draft}
    Your task is to polish the text by fixing any grammatical errors, improving the flow and sentence structure, and enhancing overall clarity.""",
    output_key="final_blog",  # This is the final output of the entire pipeline.
)

print("✅ editor_agent created.")

✅ editor_agent created.


In [16]:
root_agent = SequentialAgent(
    name="BlogPipeline",
    sub_agents=[outline_agent, writer_agent, editor_agent],
)

print("✅ Sequential Agent created.")

✅ Sequential Agent created.


In [17]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "Write a blog post about the benefits of multi-agent systems for software developers"
)


 ### Created new session: debug_session_id

User > Write a blog post about the benefits of multi-agent systems for software developers
OutlineAgent > ## Headline: Unleash Your Development Superpowers: How Multi-Agent Systems Are Revolutionizing Software

**Introduction Hook:**

Ever feel like you're juggling too many tasks, or that your software is becoming too complex to manage? What if there was a way to break down those monumental challenges into smaller, more manageable pieces, each handled by an intelligent, specialized "developer"? Enter multi-agent systems (MAS) – a paradigm shift that's empowering software developers to build more robust, scalable, and intelligent applications than ever before.

---

### Main Section 1: Deconstructing Complexity with Intelligent Autonomy

*   **Modularization on Steroids:** MAS allows you to divide your software into independent, self-contained agents. Each agent can focus on a specific task, responsibility, or domain knowledge, making the ove

---
---

### 3. Parallel Workflows - Independent Researchers



**The Problem: The Bottleneck**

The previous sequential agent is great, but it's an assembly line. Each step must wait for the previous one to finish. What if you have several tasks that are **not dependent** on each other? For example, researching three *different* topics. Running them in sequence would be slow and inefficient, creating a bottleneck where each task waits unnecessarily.

**The Solution: Concurrent Execution**

When you have independent tasks, you can run them all at the same time using a `ParallelAgent`. This agent executes all of its sub-agents concurrently, dramatically speeding up the workflow. Once all parallel tasks are complete, you can then pass their combined results to a final 'aggregator' step.

**Use Parallel when:** Tasks are independent, speed matters, and you can execute concurrently.

To learn more, check out the documentation related to [parallel agents in ADK](https://google.github.io/adk-docs/agents/workflow-agents/parallel-agents/).

#### Parallel Multi-Topic Research

Let's build a system with four agents:

1. **Tech Researcher** - Researches AI/ML news and trends
2. **Health Researcher** - Researches recent medical news and trends
3. **Finance Researcher** - Researches finance and fintech news and trends
4. **Aggregator Agent** - Combines all research findings into a single summary

**Architecture: Multi-Topic Research**

<img width="600" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/parallel-agent.png" alt="Parallel Agent" />

#### Parallel Multi-Topic Research

Let's build a system with four agents:

1. **Tech Researcher** - Researches AI/ML news and trends
2. **Health Researcher** - Researches recent medical news and trends
3. **Finance Researcher** - Researches finance and fintech news and trends
4. **Aggregator Agent** - Combines all research findings into a single summary

In [18]:
# Tech Researcher Agent : Focuses on AI/ML trends

tech_researcher = Agent(
    name = "TechResearcher",

    description = "This agent researches AI/ML news and trends.",

    model = Gemini(model="gemini-2.5-flash-lite"),

     instruction="""Research the latest AI/ML trends. Include 3 key developments,
the main companies involved, and the potential impact. Keep the report very concise (100 words).""",

    tools=[google_search],

    output_key="tech_research",  # The result of this agent will be stored in the session state with this key.
)

print("✅ tech_researcher created.")




✅ tech_researcher created.


In [19]:
# Health Researcher: Focuses on medical breakthroughs.
health_researcher = Agent(
    name="HealthResearcher",

    model=Gemini(model="gemini-2.5-flash-lite"),

    instruction="""Research recent medical breakthroughs. Include 3 significant advances,
their practical applications, and estimated timelines. Keep the report concise (100 words).""",

    tools=[google_search],

    output_key="health_research",  # The result will be stored with this key.
)

print("✅ health_researcher created.")



✅ health_researcher created.


In [20]:
# Finance Researcher: Focuses on fintech trends.
finance_researcher = Agent(
    name="FinanceResearcher",

    model=Gemini(model="gemini-2.5-flash-lite"),

    instruction="""Research current fintech trends. Include 3 key trends,
their market implications, and the future outlook. Keep the report concise (100 words).""",

    tools=[google_search],

    output_key="finance_research",  # The result will be stored with this key.
)

print("✅ finance_researcher created.")

✅ finance_researcher created.


In [22]:
# The AggregatorAgent runs *after* the parallel step to synthesize the results.
aggregator_agent = Agent(

    name="AggregatorAgent",

    model=Gemini(model="gemini-2.5-flash-lite"),

    # It uses placeholders to inject the outputs from the parallel agents, which are now in the session state.
    instruction="""Combine these three research findings into a single executive summary:

    **Technology Trends:**
    {tech_research}

    **Health Breakthroughs:**
    {health_research}

    **Finance Innovations:**
    {finance_research}

    Your summary should highlight common themes, surprising connections, and the most important key takeaways from all three reports. The final summary should be around 300 words.""",

    output_key="executive_summary",  # This will be the final output of the entire system.
)

print("✅ aggregator_agent created.")

✅ aggregator_agent created.


👉 **Then we bring the agents together under a parallel agent, which is itself nested inside of a sequential agent.**

This design ensures that the research agents run first in parallel, then once all of their research is complete, the aggregator agent brings together all of the research findings into a single report:

In [23]:
# The ParallelAgent runs all its sub-agents simultaneously.
parallel_research_team = ParallelAgent(
    name="ParallelResearchTeam",
    sub_agents=[tech_researcher, health_researcher, finance_researcher],
)

# This SequentialAgent defines the high-level workflow: run the parallel team first, then run the aggregator.
root_agent = SequentialAgent(
    name="ResearchSystem",
    sub_agents=[parallel_research_team, aggregator_agent],
)

print("✅ Parallel and Sequential Agents created.")

✅ Parallel and Sequential Agents created.


In [24]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "Run the daily executive briefing on Tech, Health, and Finance"
)


 ### Created new session: debug_session_id

User > Run the daily executive briefing on Tech, Health, and Finance
FinanceResearcher > **Tech Executive Briefing:**

The European Commission has fined social media company X $140 million for breaching the EU's Digital Services Act, citing deceptive design and lack of transparency. In broader tech, the World Economic Forum highlights the technology sector's dependence on natural resources and its environmental impact, urging for nature-positive strategies and financial opportunities in the green economy.

**Health Executive Briefing:**

US President Trump has ordered a review of the childhood vaccine schedule, questioning the number of recommended shots. This follows a panel's vote to end the hepatitis B vaccine recommendation for newborns. In digital health, an AI companion for nurses has raised $50 million in Series A funding to accelerate expansion.

**Finance Executive Briefing:**

The IMF is considering a $200 million emergency aid req

---


### Loop Workflows - The Refinement Cycle

**The Problem: One-Shot Quality**

All the workflows we've seen so far run from start to finish. The `SequentialAgent` and `ParallelAgent` produce their final output and then stop. This 'one-shot' approach isn't good for tasks that require refinement and quality control. What if the first draft of our story is bad? We have no way to review it and ask for a rewrite.

**The Solution: Iterative Refinement**

When a task needs to be improved through cycles of feedback and revision, you can use a `LoopAgent`. A `LoopAgent` runs a set of sub-agents repeatedly *until a specific condition is met or a maximum number of iterations is reached.* This creates a refinement cycle, allowing the agent system to improve its own work over and over.

**Use Loop when:** Iterative improvement is needed, quality refinement matters, or you need repeated cycles.

To learn more, check out the documentation related to [loop agents in ADK](https://google.github.io/adk-docs/agents/workflow-agents/loop-agents/).


**Architecture: Story Writing & Critique Loop**

<img width="250" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/loop-agent.png" alt="Loop Agent" />

#### Iterative Story Refinement

Let's build a system with two agents:

1. **Writer Agent** - Writes a draft of a short story
2. **Critic Agent** - Reviews and critiques the short story to suggest improvements

In [26]:
# This agent runs ONCE at the beginning to create the first draft.
initial_writer_agent = Agent(
    name="InitialWriterAgent",

    model=Gemini(model="gemini-2.5-flash-lite"),

    instruction="""Based on the user's prompt, write the first draft of a short story (around 200-300 words).
    Output only the story text, with no introduction or explanation.""",

    output_key="current_story",  # Stores the first draft in the state.
)

print("✅ initial_writer_agent created.")

✅ initial_writer_agent created.


In [27]:
# This agent's only job is to provide feedback or the approval signal. It has no tools.
critic_agent = Agent(
    name="CriticAgent",

    model=Gemini(model="gemini-2.5-flash-lite"),

    instruction="""You are a constructive story critic. Review the story provided below.
    Story: {current_story}
    Evaluate the story's plot, characters, and pacing.
    - If the story is well-written and complete, you MUST respond with the exact phrase: "APPROVED"
    - Otherwise, provide 2-3 specific, actionable suggestions for improvement.""",

    output_key="critique",  # Stores the feedback in the state.
)

print("✅ critic_agent created.")

✅ critic_agent created.


Now, we need a way for the loop to actually stop based on the critic's feedback. The `LoopAgent` itself doesn't automatically know that "APPROVED" means "stop."

We need an agent to give it an explicit signal to terminate the loop.

We do this in two parts:

1. A simple Python function that the `LoopAgent` understands as an "exit" signal.
2. An agent that can call that function when the right condition is met.

First, you'll define the `exit_loop` function:

In [30]:
# This is the function that the RefinerAgent will call to exit the loop.
def exit_loop():
    """Call this function ONLY when the critique is 'APPROVED', indicating the story is finished and no more changes are needed."""
    return {"status": "approved", "message": "Story approved. Exiting refinement loop."}


print("✅ exit_loop function created.")

✅ exit_loop function created.


To let an agent call this Python function, we wrap it in a `FunctionTool`. Then, we create a `RefinerAgent` that has this tool.

👉 **Notice its instructions:** this agent is the "brain" of the loop. It reads the `{critique}` from the `CriticAgent` and decides whether to
(1) call the `exit_loop` tool or
(2) rewrite the story.

In [31]:
# This agent refines the story based on critique OR calls the exit_loop function.
refiner_agent = Agent(
    name="RefinerAgent",

    model=Gemini(model="gemini-2.5-flash-lite"),

    instruction="""You are a story refiner. You have a story draft and critique.

    Story Draft: {current_story}
    Critique: {critique}

    Your task is to analyze the critique.
    - IF the critique is EXACTLY "APPROVED", you MUST call the `exit_loop` function and nothing else.
    - OTHERWISE, rewrite the story draft to fully incorporate the feedback from the critique.""",

    output_key="current_story",  # It overwrites the story with the new, refined version.

    tools=[FunctionTool(exit_loop)],  # The tool is now correctly initialized with the function reference.
)

print("✅ refiner_agent created.")

✅ refiner_agent created.


Then we bring the agents together under a loop agent, which is itself nested inside of a sequential agent.

This design ensures that the system first produces an initial story draft, then the refinement loop runs up to the specified number of `max_iterations`:

In [32]:
# The LoopAgent contains the agents that will run repeatedly: Critic -> Refiner.
story_refinement_loop = LoopAgent(
    name="StoryRefinementLoop",
    sub_agents=[critic_agent, refiner_agent],
    max_iterations=3,  # Prevents infinite loops
)

# The root agent is a SequentialAgent that defines the overall workflow: Initial Write -> Refinement Loop.
root_agent = SequentialAgent(
    name="StoryPipeline",
    sub_agents=[initial_writer_agent, story_refinement_loop],
)

print("✅ Loop and Sequential Agents created.")

✅ Loop and Sequential Agents created.


In [33]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "Write a short story about a lighthouse keeper who discovers a mysterious, glowing map"
)


 ### Created new session: debug_session_id

User > Write a short story about a lighthouse keeper who discovers a mysterious, glowing map
InitialWriterAgent > The salt spray was Elara’s constant companion, a damp kiss on her weathered cheeks as she climbed the spiral stairs. For twenty years, the beam of the Serpent’s Tooth lighthouse had been her only conversation, its rhythmic sweep a lullaby against the roaring sea. Tonight, however, something was different.

While polishing the brass fittings in the lantern room, her cloth snagged on a loose panel near the base of the great lens. Curiosity piqued, she pried it open. Inside, nestled amongst a tangle of ancient wires, lay a rolled parchment. It wasn’t ordinary paper; it felt strangely cool, almost alive.

Unfurling it, Elara gasped. The parchment glowed with an ethereal, aquamarine light. Intricate lines, shimmering like captured starlight, crisscrossed its surface, forming what looked undeniably like a map. But it depicted no land s

### Summary - Choosing the Right Pattern

#### Decision Tree: Which Workflow Pattern?

<img width="1000" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/agent-decision-tree.png" alt="Agent Decision Tree" />

### Quick Reference Table

| Pattern | When to Use | Example | Key Feature |
|---------|-------------|---------|-------------|
| **LLM-based (sub_agents)** | Dynamic orchestration needed | Research + Summarize | LLM decides what to call |
| **Sequential** | Order matters, linear pipeline | Outline → Write → Edit | Deterministic order |
| **Parallel** | Independent tasks, speed matters | Multi-topic research | Concurrent execution |
| **Loop** | Iterative improvement needed | Writer + Critic refinement | Repeated cycles |

We used `SequentialAgent`, `ParallelAgent`, and `LoopAgent` to create deterministic workflows, and we even used an LLM as a 'manager' to make dynamic decisions. we also mastered the "plumbing" by using `output_key` to pass state between agents and make them collaborative.